# Day 35: Audit RAG Performance & "Lost in the Middle"

Welcome to Day 35 of the AI Engineering Mastery program! Today, we dive into auditing Retrieval-Augmented Generation (RAG) performance, focusing specifically on the **"Lost in the Middle"** phenomenon.

## Core Theory: What is "Lost in the Middle"?

When LLMs are presented with long contexts (like multiple retrieved documents in a RAG pipeline), they are remarkably good at extracting information located at the very **beginning** or the very **end** of the context. However, their performance drops significantly when the relevant information is buried in the **middle** of the context.

This happens because of how attention mechanisms in Transformers work. The model tends to heavily weight the start of the prompt (the instructions and the first few docs) and the end of the prompt (the most recent tokens seen before generation).

**Why does this matter in RAG?**
Standard vector search (like Qdrant or FAISS) returns documents sorted by relevance (cosine distance). If you retrieve 10 documents, the most relevant are first, and the least relevant are last. If you inject them into the prompt in this exact order:
1. Doc 1 (Highest relevance) - *Model pays attention*
2. Doc 2 (High relevance)
...
5. Doc 5 (Medium relevance) - *Model loses focus (Lost in the middle)*
...
10. Doc 10 (Lowest relevance) - *Model pays attention*

The LLM is paying the most attention to your best document (good) and your worst document (bad), while ignoring the decent documents in the middle.

**The Solution: Reordering (The "How")**
To mitigate this, we can intercept the retrieved documents and reorder them before sending them to the LLM. A common technique is to alternate placing the most relevant documents at the beginning and the end of the context window.

Example reordering of 5 documents (1 is most relevant, 5 is least):
*Original:* `[1, 2, 3, 4, 5]`
*Reordered:* `[1, 3, 5, 4, 2]`

This ensures the most crucial information is placed at the extremities where the LLM is most likely to "see" it.

## 1. Setup & Baseline Retrieval (Mock)

Let's set up a mock scenario where we have retrieved 10 documents, sorted by relevance (1 being the most relevant, 10 being the least).

In [1]:
from typing import List
from langchain_core.documents import Document

# Simulate documents returned from a Vector DB, sorted by descending relevance.
# doc_1 is the most relevant, doc_10 is the least relevant.
retrieved_docs: List[Document] = [
    Document(page_content=f"Doc {i}", metadata={"relevance_rank": i}) 
    for i in range(1, 11)
]

print("Original Retrieval Order (Sorted by Relevance):")
for doc in retrieved_docs:
    print(f"- {doc.page_content}")

Original Retrieval Order (Sorted by Relevance):
- Doc 1
- Doc 2
- Doc 3
- Doc 4
- Doc 5
- Doc 6
- Doc 7
- Doc 8
- Doc 9
- Doc 10


## 2. Implementing `LongContextReorder`

Instead of relying on third-party integrations that might be unstable, we will write a clean, robust Python function to handle the alternating reordering logic.

In [2]:
def reorder_documents(documents: List[Document]) -> List[Document]:
    """
    Reorders documents to mitigate the 'Lost in the Middle' effect.
    Places the most relevant documents at the beginning and end of the list.
    
    Args:
        documents: A list of Documents, assumed to be sorted by relevance 
                   (most relevant first).
                   
    Returns:
        A list of reordered Documents.
    """
    if not documents:
        return []

    # Create a copy to avoid mutating the original list
    docs = list(documents)
    
    # Reverse the list so the least relevant are at the beginning.
    # Original: [1, 2, 3, 4, 5] -> Reversed: [5, 4, 3, 2, 1]
    docs.reverse()
    
    reordered_docs: List[Document] = []
    
    # Alternate taking from the front and back of the reversed list
    # For [5, 4, 3, 2, 1]:
    # Iter 1 (i=0, doc=5): append -> [5]
    # Iter 2 (i=1, doc=4): insert(0) -> [4, 5]
    # Iter 3 (i=2, doc=3): append -> [4, 5, 3]
    # Iter 4 (i=3, doc=2): insert(0) -> [2, 4, 5, 3]
    # Iter 5 (i=4, doc=1): append -> [2, 4, 5, 3, 1]
    
    for i, doc in enumerate(docs):
        if i % 2 == 1:
            # Odd index in reversed list -> Insert at the beginning
            reordered_docs.insert(0, doc)
        else:
            # Even index in reversed list -> Append at the end
            reordered_docs.append(doc)
            
    return reordered_docs

reordered = reorder_documents(retrieved_docs)

print("\nReordered Document Order (Best docs at edges):")
for doc in reordered:
    print(f"- {doc.page_content}")


Reordered Document Order (Best docs at edges):
- Doc 1
- Doc 3
- Doc 5
- Doc 7
- Doc 9
- Doc 10
- Doc 8
- Doc 6
- Doc 4
- Doc 2


### Medium Implementation: Object-Oriented Approach

Encapsulating this logic into a class ensures better state management and cleaner integration into larger systems. It also allows us to easily swap out or extend reordering strategies.

In [3]:
class DocumentReorderer:
    """Handles document reordering strategies."""
    
    def reorder_lost_in_middle(self, documents: List[Document]) -> List[Document]:
        """Reorders documents to place most relevant at the extremes."""
        if not documents:
            return []
        
        docs = list(documents)
        docs.reverse()
        
        reordered_docs: List[Document] = []
        for i, doc in enumerate(docs):
            if i % 2 == 1:
                reordered_docs.insert(0, doc)
            else:
                reordered_docs.append(doc)
        return reordered_docs

reorderer = DocumentReorderer()
medium_reordered = reorderer.reorder_lost_in_middle(retrieved_docs)
print("Medium Implementation Output:")
for doc in medium_reordered:
    print(f"- {doc.page_content}")

Medium Implementation Output:
- Doc 1
- Doc 3
- Doc 5
- Doc 7
- Doc 9
- Doc 10
- Doc 8
- Doc 6
- Doc 4
- Doc 2


### Advanced Implementation: Production-Ready with AI Security & Validation

In production, we need rigorous type checking, robust error handling, and considerations for AI Security. For instance, if documents contain PII, we might want to log or handle them differently, but at the very least we need to ensure the system doesn't crash if unexpected data is passed in. We also want to validate inputs rigorously using `pydantic`.

In [4]:
import logging
from typing import List, Sequence, Any
from pydantic import BaseModel, ValidationError, Field, field_validator
from langchain_core.documents import Document

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

class ReorderRequest(BaseModel):
    """Pydantic model for validating reorder requests."""
    documents: List[Any] = Field(..., description="List of documents to reorder")
    
    @field_validator('documents')
    @classmethod
    def validate_docs(cls, v):
        if not all(hasattr(doc, 'page_content') for doc in v):
            raise ValueError("All elements in documents must have 'page_content' attribute.")
        return v
    
    class Config:
        arbitrary_types_allowed = True

class AdvancedDocumentReorderer:
    """
    Production-grade document reorderer with logging and error handling.
    """
    def __init__(self, strategy: str = "lost_in_middle"):
        if strategy not in ["lost_in_middle"]:
            raise ValueError(f"Unsupported strategy: {strategy}")
        self.strategy = strategy
        logger.info(f"Initialized AdvancedDocumentReorderer with strategy: {self.strategy}")

    def reorder(self, documents: Sequence[Document]) -> List[Document]:
        """
        Reorders documents based on the configured strategy.
        
        Args:
            documents: A sequence of Langchain Document objects.
            
        Returns:
            A new list of reordered Documents.
        """
        try:
            # Validate input using Pydantic
            validated_request = ReorderRequest(documents=list(documents))
            docs = validated_request.documents
        except ValidationError as e:
            logger.error(f"Input validation failed: {e}")
            raise ValueError(f"Invalid input for reordering: {e}")

        if not docs:
            logger.warning("Received empty document list for reordering.")
            return []

        logger.info(f"Reordering {len(docs)} documents using '{self.strategy}' strategy.")
        
        docs.reverse()
        
        reordered_docs: List[Document] = []
        for i, doc in enumerate(docs):
            if i % 2 == 1:
                reordered_docs.insert(0, doc)
            else:
                reordered_docs.append(doc)
                
        return reordered_docs

# Example Usage
try:
    adv_reorderer = AdvancedDocumentReorderer()
    adv_reordered = adv_reorderer.reorder(retrieved_docs)
    print("\nAdvanced Implementation Output:")
    for doc in adv_reordered:
        print(f"- {doc.page_content}")
except Exception as e:
    logger.error(f"Reordering failed: {e}")


/tmp/ipykernel_69744/2021665659.py:10: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ReorderRequest(BaseModel):
INFO: Initialized AdvancedDocumentReorderer with strategy: lost_in_middle


INFO: Reordering 10 documents using 'lost_in_middle' strategy.



Advanced Implementation Output:
- Doc 1
- Doc 3
- Doc 5
- Doc 7
- Doc 9
- Doc 10
- Doc 8
- Doc 6
- Doc 4
- Doc 2


## Common Pitfalls in Production

1. **Blindly applying reordering:** If you only retrieve 2-3 short documents, the context window is small enough that "Lost in the Middle" doesn't occur. Reordering here just adds overhead. Only apply this when context lengths exceed a certain threshold (e.g., >3000 tokens or >5 documents).
2. **Ignoring Document Length:** The reordering algorithm assumes documents are roughly the same length. If `Doc 1` is 50 tokens and `Doc 2` is 4000 tokens, placing `Doc 2` at the end might push `Doc 1` out of the critical "start" zone entirely. You must chunk documents uniformly.
3. **Over-Retrieval:** Trying to solve bad retrieval by retrieving *more* documents (e.g., top 50) and then reordering them often degrades performance regardless of reordering. Always optimize base retrieval quality first.
4. **Not measuring the impact:** You must audit RAG performance (e.g., using Ragas or LLM-as-a-judge) before and after implementing reordering to ensure it actually improves your specific task.

## Practical Lab / Homework: End-to-End Qdrant Integration

**Your Task:**
Instead of working with mock lists, you will now integrate this into a functional RAG setup using `Qdrant` and `Langchain`.

1. Initialize an in-memory `Qdrant` vector store.
2. Use `FakeEmbeddings` (size 1536) to simulate a real embedding model (like OpenAI).
3. Insert 10 dummy documents into the vector store.
4. Perform a similarity search to retrieve the top 5 documents.
5. Apply your `AdvancedDocumentReorderer` to the retrieved documents.
6. Print the original retrieval order and the reordered results.

*Challenge:* Record a brief async video walkthrough of your design decisions, explaining why you chose to place the reordering logic where you did within the pipeline.

In [5]:
from langchain_core.documents import Document
from langchain_core.embeddings.fake import FakeEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# 1. Setup in-memory Qdrant client
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="lab_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

# 2. Setup Fake Embeddings and Vector Store
embeddings = FakeEmbeddings(size=1536)
vector_store = QdrantVectorStore(
    client=client,
    collection_name="lab_collection",
    embedding=embeddings
)

# 3. Insert Dummy Documents
dummy_docs = [
    Document(page_content=f"Crucial piece of info {i}", metadata={"id": i})
    for i in range(1, 11)
]
vector_store.add_documents(dummy_docs)

# 4. Perform Similarity Search
# We query for something to get results back. 
# Note: FakeEmbeddings will return random vectors, so order is effectively random, 
# but we will still see the reordering logic applied to the list.
query = "Find me crucial information"
retrieved_results = vector_store.similarity_search(query, k=5)

print("Original Retrieval Order (from Qdrant):")
for i, doc in enumerate(retrieved_results):
    print(f"{i+1}. {doc.page_content}")

# 5. Apply Reordering
lab_reorderer = AdvancedDocumentReorderer()
reordered_results = lab_reorderer.reorder(retrieved_results)

print("\nReordered Results (Lost-in-the-middle mitigated):")
for i, doc in enumerate(reordered_results):
    print(f"{i+1}. {doc.page_content}")

INFO: Initialized AdvancedDocumentReorderer with strategy: lost_in_middle


INFO: Reordering 5 documents using 'lost_in_middle' strategy.


Original Retrieval Order (from Qdrant):
1. Crucial piece of info 6
2. Crucial piece of info 3
3. Crucial piece of info 1
4. Crucial piece of info 4
5. Crucial piece of info 9

Reordered Results (Lost-in-the-middle mitigated):
1. Crucial piece of info 3
2. Crucial piece of info 4
3. Crucial piece of info 9
4. Crucial piece of info 1
5. Crucial piece of info 6


## Reference Links

- [Lost in the Middle: How Language Models Use Long Contexts (Paper)](https://arxiv.org/abs/2307.03172)
- [LangChain Document Transformers Documentation](https://python.langchain.com/docs/modules/data_connection/document_transformers/)
- [Qdrant Documentation](https://qdrant.tech/documentation/)